In [ ]:
# Mount Google Drive and set up repo
from google.colab import drive
drive.mount('/content/drive')

import subprocess, os
REPO_PATH = '/content/lung-nodule-fusion-xai'
if not os.path.exists(REPO_PATH):
    subprocess.run(['git', 'clone', 'https://github.com/YOUR_USERNAME/lung-nodule-fusion-xai.git', REPO_PATH], check=True)
os.chdir(REPO_PATH)


In [ ]:
# Install required packages (skips torch — Colab has it pre-installed)
!pip install -q pylidc==0.2.2 SimpleITK==2.3.1 nibabel==5.2.1 pyradiomics==3.1.0 \
               scikit-learn==1.5.0 xgboost==2.0.3 pandas==2.2.2 pyarrow==16.1.0


In [ ]:
# Configure pylidc to find LIDC-IDRI data
# LIDC-IDRI should be downloaded to /content/drive/MyDrive/Kanker Kanker apa yg Kanker/lidc_idri/
import pylidc as pl
import configparser, os

cfg = configparser.ConfigParser()
cfg['pylidc'] = {'dicom_path': '/content/drive/MyDrive/Kanker Kanker apa yg Kanker/lidc_idri'}
with open(os.path.expanduser('~/.pylidcrc'), 'w') as f:
    cfg.write(f)
print("pylidc configured")


In [ ]:
import sys
sys.path.insert(0, '/content/lung-nodule-fusion-xai')

from src.data_loading.lidc_loader import load_and_split

df = load_and_split(
    lidc_path='/content/drive/MyDrive/Kanker Kanker apa yg Kanker/lidc_idri',
    interim_path='/content/drive/MyDrive/lung_nodule_interim',
    processed_path='/content/drive/MyDrive/lung_nodule_processed',
    n_folds=5,
    seed=42,
)
print(f"Total nodules: {len(df)}")
print(f"Malignant: {(df.label==1).sum()}, Benign: {(df.label==0).sum()}")
print(df.head())


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

fold_counts = df.groupby(['fold', 'label']).size().unstack(fill_value=0)
fold_counts.plot(kind='bar', figsize=(8, 4), title='Fold Distribution by Label')
plt.xlabel('Fold')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/lung_nodule_processed/fold_distribution.png', dpi=150)
plt.show()
print("Fold distribution saved")


In [ ]:
# ============================================================
# LUNA16 PIPELINE (alternative to LIDC-IDRI DICOM above)
# Run this section INSTEAD of cells 2-4 if using LUNA16 data
# ============================================================
DRIVE_BASE = '/content/drive/MyDrive/Kanker Kanker apa yg Kanker'
SUBSET_DIR = '/content/luna16'   # local extraction target
SUBSET_IDS = [0, 1, 2, 3]        # adjust: which subsets to use (0-9)
                                  # each subset ~6-7 GB; start with 0-3


In [ ]:
# Step L1: Extract subset zips from Drive
from src.data_loading.luna16_loader import extract_subsets

extract_subsets(
    output_dir=SUBSET_DIR,
    subset_ids=SUBSET_IDS,
    drive_part1=f'{DRIVE_BASE}/luna16 part 1',
    drive_part2=f'{DRIVE_BASE}/luna16 part 2',
)
print('Extraction complete')


In [ ]:
# Step L2: Build LUNA16 labels DataFrame
# Malignancy labels fetched from LIDC-IDRI XML via pylidc
# (requires pylidc configured — same ~/.pylidcrc as LIDC pipeline above)
import configparser, os
cfg = configparser.ConfigParser()
cfg['pylidc'] = {'dicom_path': f'{DRIVE_BASE}/lidc_idri'}
with open(os.path.expanduser('~/.pylidcrc'), 'w') as f:
    cfg.write(f)

from src.data_loading.luna16_loader import load_and_split_luna16

CANDIDATES_CSV = f'{DRIVE_BASE}/luna16 part 1/candidates.csv'
METADATA_CSV   = f'{DRIVE_BASE}/metadata/metadata.csv'

df_luna = load_and_split_luna16(
    subset_dir=SUBSET_DIR,
    candidates_csv=CANDIDATES_CSV,
    metadata_csv=METADATA_CSV,
    lidc_path=f'{DRIVE_BASE}/lidc_idri',
    output_dir='/content/drive/MyDrive/lung_nodule_interim_luna16',
    processed_path='/content/drive/MyDrive/lung_nodule_processed_luna16',
    n_folds=5,
    seed=42,
)
print(f'Total nodules: {len(df_luna)}')
print(f'Malignant: {(df_luna.label==1).sum()}, Benign: {(df_luna.label==0).sum()}')
print(df_luna.head())


In [ ]:
# Step L3: Save fold distribution plot
import matplotlib.pyplot as plt

fold_counts = df_luna.groupby(['fold', 'label']).size().unstack(fill_value=0)
fold_counts.plot(kind='bar', figsize=(8, 4),
                 title='LUNA16 Fold Distribution by Label')
plt.xlabel('Fold')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/lung_nodule_processed_luna16/fold_distribution.png',
            dpi=150)
plt.show()
